# DemandCast - Phase 0: Data Audit

**Dataset:** DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS
**Source:** Mendeley Data, Version 5, DOI: 10.17632/8gx2fvg2k6.5
**License:** CC BY 4.0
**Citation:** Constante, Fabian; Silva, Fernando; Pereira, Antonio (2019)

This notebook performs a comprehensive audit of the raw dataset to understand its structure, quality, and suitability for demand forecasting.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from pathlib import Path
import yaml
import json
from datetime import datetime

# Load config
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

raw_path = Path(config['paths']['raw_data'])
transactions_file = raw_path / config['raw_files']['transactions']
dict_file = raw_path / config['raw_files']['data_dictionary']

print(f"Transactions file: {transactions_file}")
print(f"Data dictionary file: {dict_file}")
print(f"Transactions exists: {transactions_file.exists()}")
print(f"Dictionary exists: {dict_file.exists()}")

In [ ]:
# Load the data with appropriate encoding
# The dataset likely needs ISO-8859-1 encoding
encodings_to_try = ['iso-8859-1', 'latin-1', 'cp1252', 'utf-8']

df = None
used_encoding = None

for enc in encodings_to_try:
    try:
        df = pd.read_csv(transactions_file, encoding=enc, low_memory=False)
        used_encoding = enc
        print(f"Successfully loaded with encoding: {enc}")
        break
    except UnicodeDecodeError as e:
        print(f"Failed with {enc}: {e}")
    except Exception as e:
        print(f"Error with {enc}: {e}")

if df is None:
    raise ValueError("Could not load the dataset with any of the tried encodings")

print(f"\nShape: {df.shape}")
print(f"Encoding used: {used_encoding}")

In [ ]:
# Basic info
print("=== DATASET INFO ===")
df.info(verbose=True, show_counts=True)

print("\n=== COLUMN NAMES ===")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

In [ ]:
# First few rows
print("=== FIRST 5 ROWS ===")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 50)
display(df.head())

In [ ]:
# Missing values analysis
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing_df.to_string())

print(f"\nTotal rows: {len(df):,}")
print(f"Columns with missing: {len(missing_df)}")
print(f"Columns complete: {df.shape[1] - len(missing_df)}")

In [ ]:
# Duplicate rows
print("=== DUPLICATE ANALYSIS ===")
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes:,} ({exact_dupes/len(df)*100:.2f}%)")

# Check for duplicate transaction lines (if there's an ID column)
id_cols = [c for c in df.columns if 'id' in c.lower() or 'order' in c.lower()]
print(f"Potential ID columns: {id_cols}")

if id_cols:
    for col in id_cols:
        dupe_count = df.duplicated(subset=[col]).sum()
        print(f"  Duplicates on {col}: {dupe_count:,}")

In [ ]:
# Data types and unique counts
print("=== COLUMN DETAILS ===")
details = []
for col in df.columns:
    dtype = str(df[col].dtype)
    n_unique = df[col].nunique()
    n_missing = df[col].isnull().sum()
    sample_vals = df[col].dropna().unique()[:5]
    details.append({
        'column': col,
        'dtype': dtype,
        'n_unique': n_unique,
        'n_missing': n_missing,
        'missing_pct': round(n_missing/len(df)*100, 2),
        'sample_values': str(sample_vals)
    })

details_df = pd.DataFrame(details)
display(details_df)

In [ ]:
# Load data dictionary
dict_df = None
for enc in encodings_to_try:
    try:
        dict_df = pd.read_csv(dict_file, encoding=enc)
        print(f"Dictionary loaded with encoding: {enc}")
        break
    except Exception as e:
        print(f"Failed with {enc}: {e}")

if dict_df is not None:
    print("=== DATA DICTIONARY ===")
    display(dict_df)
else:
    print("Could not load data dictionary")

In [ ]:
# Date column analysis
print("=== DATE COLUMN ANALYSIS ===")
date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
print(f"Potential date columns: {date_cols}")

for col in date_cols:
    print(f"\n--- {col} ---")
    print(f"  Dtype: {df[col].dtype}")
    print(f"  Non-null: {df[col].notna().sum():,}")
    print(f"  Unique: {df[col].nunique():,}")
    sample = df[col].dropna().unique()[:10]
    print(f"  Sample values: {sample}")

In [ ]:
# Parse dates and check coverage
# Try to identify the main transaction date column
date_col_candidates = ['Order_Date', 'order_date', 'Date', 'date', 'Transaction_Date', 'transaction_date']
date_col = None
for cand in date_col_candidates:
    if cand in df.columns:
        date_col = cand
        break

if date_col is None and date_cols:
    date_col = date_cols[0]

if date_col:
    print(f"Using date column: {date_col}")
    # Try parsing
    parsed = pd.to_datetime(df[date_col], errors='coerce')
    print(f"  Successfully parsed: {parsed.notna().sum():,} / {len(df):,}")
    print(f"  Failed to parse: {parsed.isna().sum():,}")
    
    if parsed.notna().any():
        print(f"  Min date: {parsed.min()}")
        print(f"  Max date: {parsed.max()}")
        print(f"  Date range: {(parsed.max() - parsed.min()).days} days")
        
        # Check for partial final period
        # Get the last month/week of data
        max_date = parsed.max()
        # Check how many days in the final week/month
        final_week_start = max_date - pd.Timedelta(days=6)
        final_week_data = parsed[(parsed >= final_week_start) & (parsed <= max_date)]
        print(f"  Days in final week: {final_week_data.notna().sum()}")
        print(f"  Unique dates in final week: {final_week_data.nunique()}")
        
        # Monthly breakdown of final month
        final_month_start = max_date.replace(day=1)
        final_month_data = parsed[(parsed >= final_month_start) & (parsed <= max_date)]
        print(f"  Days in final month: {final_month_data.notna().sum()}")
        print(f"  Unique dates in final month: {final_month_data.nunique()}")
        
        # Day of week distribution in final week
        if len(final_week_data.dropna()) > 0:
            dow_counts = final_week_data.dropna().dt.day_name().value_counts().sort_index()
            print(f"  Day of week in final week:\n{dow_counts}")
else:
    print("No date column found")

In [ ]:
# Categorical column analysis
print("=== CATEGORICAL ANALYSIS ===")
cat_cols = [c for c in df.columns if df[c].dtype == 'object' or df[c].dtype.name == 'category']
print(f"Object/category columns: {cat_cols}")

key_cats = ['Category', 'category', 'Market', 'market', 'Region', 'region', 'Country', 'country',
            'Customer', 'customer', 'Product', 'product', 'Order', 'order', 'Warehouse', 'warehouse']

for col in cat_cols:
    if any(k.lower() in col.lower() for k in ['category', 'market', 'region', 'country', 'customer', 'product', 'warehouse']):
        print(f"\n--- {col} ---")
        print(f"  Unique values: {df[col].nunique():,}")
        vc = df[col].value_counts().head(20)
        print(f"  Top 20:\n{vc}")

In [ ]:
# Numeric column analysis
print("=== NUMERIC ANALYSIS ===")
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {num_cols}")

for col in num_cols:
    print(f"\n--- {col} ---")
    desc = df[col].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    print(desc)
    print(f"  Zeros: {(df[col] == 0).sum():,}")
    print(f"  Negatives: {(df[col] < 0).sum():,}")

In [ ]:
# Target variable candidates
print("=== TARGET VARIABLE CANDIDATES ===")
target_candidates = [c for c in df.columns if any(k in c.lower() for k in ['quantity', 'qty', 'amount', 'sales', 'demand', 'units', 'volume'])]
print(f"Potential target columns: {target_candidates}")

for col in target_candidates:
    print(f"\n--- {col} ---")
    print(f"  Dtype: {df[col].dtype}")
    print(f"  Range: {df[col].min()} to {df[col].max()}")
    print(f"  Mean: {df[col].mean():.2f}")
    print(f"  Median: {df[col].median():.2f}")
    print(f"  Zeros: {(df[col] == 0).sum():,} ({(df[col] == 0).sum()/len(df)*100:.1f}%)")
    print(f"  Negatives: {(df[col] < 0).sum():,}")
    print(f"  Missing: {df[col].isnull().sum():,}")

In [ ]:
# Transaction grain analysis
print("=== TRANSACTION GRAIN ===")
# Check if each row is a unique transaction line
id_cols = [c for c in df.columns if 'id' in c.lower() or 'order' in c.lower() or 'line' in c.lower()]
print(f"Potential ID columns: {id_cols}")

if id_cols:
    for col in id_cols:
        print(f"  {col}: {df[col].nunique():,} unique, {df[col].isnull().sum():,} missing")
    
    # Check combination uniqueness
    if len(id_cols) >= 2:
        combo = df[id_cols[:2]].drop_duplicates()
        print(f"  Unique combinations of first 2 ID cols: {len(combo):,}")
        print(f"  Total rows: {len(df):,}")

In [ ]:
# Aggregation level analysis
print("=== AGGREGATION LEVEL ANALYSIS ===")

# Find category, market, region columns
cat_col = None
market_col = None
region_col = None

for col in df.columns:
    cl = col.lower()
    if 'category' in cl and 'product' not in cl and cat_col is None:
        cat_col = col
    elif 'market' in cl and market_col is None:
        market_col = col
    elif 'region' in cl and region_col is None:
        region_col = col

print(f"Category column: {cat_col}")
print(f"Market column: {market_col}")
print(f"Region column: {region_col}")

target_col = 'Order_Item_Quantity'  # Based on DataCo dataset
if target_col not in df.columns:
    # Find the actual quantity column
    for col in target_candidates:
        if 'quantity' in col.lower() or 'qty' in col.lower():
            target_col = col
            break

print(f"Target column: {target_col}")

if cat_col and target_col in df.columns:
    print(f"\n--- By {cat_col} ---")
    cat_stats = df.groupby(cat_col)[target_col].agg(['count', 'sum', 'mean', 'median', 'std']).round(2)
    cat_stats.columns = ['n_transactions', 'total_qty', 'avg_qty', 'median_qty', 'std_qty']
    display(cat_stats.sort_values('total_qty', ascending=False))

In [ ]:
# Time series analysis at different aggregation levels
print("=== TIME SERIES AGGREGATION ANALYSIS ===")

if date_col and target_col in df.columns:
    # Parse dates
    df['_parsed_date'] = pd.to_datetime(df[date_col], errors='coerce')
    
    # Overall daily
    daily = df.dropna(subset=['_parsed_date']).groupby('_parsed_date')[target_col].sum()
    print(f"Overall daily series: {len(daily)} days")
    print(f"  Date range: {daily.index.min()} to {daily.index.max()}")
    print(f"  Mean daily demand: {daily.mean():.2f}")
    print(f"  Zero days: {(daily == 0).sum()}")
    
    # Weekly (Monday start)
    weekly = daily.resample('W-MON').sum()
    print(f"\nOverall weekly series: {len(weekly)} weeks")
    print(f"  Mean weekly demand: {weekly.mean():.2f}")
    print(f"  Zero weeks: {(weekly == 0).sum()}")
    
    # Monthly
    monthly = daily.resample('MS').sum()
    print(f"\nOverall monthly series: {len(monthly)} months")
    print(f"  Mean monthly demand: {monthly.mean():.2f}")
    print(f"  Zero months: {(monthly == 0).sum()}")
    
    # By category weekly
    if cat_col:
        cat_weekly = df.dropna(subset=['_parsed_date']).groupby([cat_col, pd.Grouper(key='_parsed_date', freq='W-MON')])[target_col].sum().unstack(fill_value=0)
        print(f"\nCategory-weekly matrix: {cat_weekly.shape[0]} categories x {cat_weekly.shape[1]} weeks")
        
        # Series length per category
        cat_lengths = (cat_weekly > 0).sum(axis=1)
        print(f"  Weeks with demand per category:\n{cat_lengths.describe()}")
        print(f"  Categories with < 26 weeks data: {(cat_lengths < 26).sum()}")
        print(f"  Categories with < 52 weeks data: {(cat_lengths < 52).sum()}")
    
    # By market weekly
    if market_col:
        mkt_weekly = df.dropna(subset=['_parsed_date']).groupby([market_col, pd.Grouper(key='_parsed_date', freq='W-MON')])[target_col].sum().unstack(fill_value=0)
        print(f"\nMarket-weekly matrix: {mkt_weekly.shape[0]} markets x {mkt_weekly.shape[1]} weeks")
        mkt_lengths = (mkt_weekly > 0).sum(axis=1)
        print(f"  Weeks with demand per market:\n{mkt_lengths.describe()}")

In [ ]:
# Intermittent demand analysis
print("=== INTERMITTENT DEMAND ANALYSIS ===")

if date_col and target_col in df.columns and cat_col:
    # Compute ADI (Average Demand Interval) and CV^2 for each category
    cat_series = df.dropna(subset=['_parsed_date']).groupby([cat_col, pd.Grouper(key='_parsed_date', freq='W-MON')])[target_col].sum()
    
    intermittent_stats = []
    for cat in cat_series.index.get_level_values(0).unique():
        series = cat_series.loc[cat].values
        non_zero = series[series > 0]
        zero_periods = (series == 0).sum()
        total_periods = len(series)
        
        if len(non_zero) > 1:
            adi = total_periods / len(non_zero)
            cv2 = (non_zero.std() / non_zero.mean()) ** 2 if non_zero.mean() > 0 else np.nan
        else:
            adi = total_periods
            cv2 = np.nan
        
    intermittent_stats.append({
        'category': cat,
        'total_periods': total_periods,
        'non_zero_periods': len(non_zero),
        'zero_periods': zero_periods,
        'adi': round(adi, 2),
        'cv2': round(cv2, 4) if not np.isnan(cv2) else np.nan,
        'mean_demand': round(non_zero.mean(), 2) if len(non_zero) > 0 else 0,
        'intermittent_adi': adi > 1.32,
        'intermittent_cv2': cv2 > 0.49 if not np.isnan(cv2) else False
    })
    
    intermittent_df = pd.DataFrame(intermittent_stats)
    intermittent_df = intermittent_df.sort_values('adi', ascending=False)
    display(intermittent_df)
    
    print(f"\nCategories with ADI > 1.32: {(intermittent_df['intermittent_adi']).sum()} / {len(intermittent_df)}")
    print(f"Categories with CV^2 > 0.49: {(intermittent_df['intermittent_cv2']).sum()} / {len(intermittent_df)}")
    print(f"Both criteria (intermittent): {((intermittent_df['intermittent_adi']) & (intermittent_df['intermittent_cv2'])).sum()}")

In [ ]:
# PII column check
print("=== PII COLUMN CHECK ===")
pii_keywords = ['email', 'name', 'password', 'address', 'street', 'zip', 'postal', 'phone', 'ssn', 'credit', 'card']
pii_cols = []
for col in df.columns:
    cl = col.lower()
    if any(kw in cl for kw in pii_keywords):
        pii_cols.append(col)

if pii_cols:
    print(f"Potential PII columns found: {pii_cols}")
    for col in pii_cols:
        print(f"  {col}: {df[col].dtype}, {df[col].nunique()} unique, sample: {df[col].dropna().unique()[:3]}")
else:
    print("No obvious PII columns found by keyword search")

# Also check for high-cardinality identifier columns that should be dropped
id_like = []
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() > len(df) * 0.5:
        id_like.append(col)
    elif df[col].dtype in ['int64', 'float64'] and df[col].nunique() == len(df):
        id_like.append(col)

print(f"\nHigh-cardinality columns (likely IDs): {id_like}")

In [ ]:
# Shipping/delivery columns check
print("=== SHIPPING/DELIVERY COLUMNS ===")
ship_cols = [c for c in df.columns if any(k in c.lower() for k in ['ship', 'delivery', 'lead', 'transit', 'days'])]
print(f"Shipping-related columns: {ship_cols}")

for col in ship_cols:
    print(f"\n--- {col} ---")
    print(f"  Dtype: {df[col].dtype}")
    if df[col].dtype in ['int64', 'float64']:
        print(f"  Range: {df[col].min()} to {df[col].max()}")
        print(f"  Mean: {df[col].mean():.2f}")
        print(f"  Median: {df[col].median():.2f}")
    else:
        vc = df[col].value_counts().head(10)
        print(f"  Top values:\n{vc}")

In [ ]:
# Summary and Recommendations
print("=== AUDIT SUMMARY ===")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print(f"Encoding used: {used_encoding}")
print(f"Date column: {date_col}")
print(f"Target column: {target_col}")
print(f"Category column: {cat_col}")
print(f"Market column: {market_col}")
print(f"Region column: {region_col}")

if date_col and target_col in df.columns:
    df['_parsed_date'] = pd.to_datetime(df[date_col], errors='coerce')
    valid_dates = df['_parsed_date'].notna()
    print(f"Valid dates: {valid_dates.sum():,} / {len(df):,}")
    if valid_dates.any():
        print(f"Date range: {df.loc[valid_dates, '_parsed_date'].min()} to {df.loc[valid_dates, '_parsed_date'].max()}")
        
    # Final period check
    max_date = df.loc[valid_dates, '_parsed_date'].max()
    final_week_start = max_date - pd.Timedelta(days=6)
    final_week = df[(df['_parsed_date'] >= final_week_start) & (df['_parsed_date'] <= max_date)]
    print(f"Final week rows: {len(final_week):,}")
    print(f"Final week unique dates: {final_week['_parsed_date'].nunique()}")
    
    # Check if final month is complete
    final_month_start = max_date.replace(day=1)
    if final_month_start.month == 12:
        next_month = final_month_start.replace(year=final_month_start.year+1, month=1)
    else:
        next_month = final_month_start.replace(month=final_month_start.month+1)
    expected_days = (next_month - final_month_start).days
    final_month = df[(df['_parsed_date'] >= final_month_start) & (df['_parsed_date'] < next_month)]
    actual_days = final_month['_parsed_date'].nunique()
    print(f"Final month: {actual_days} / {expected_days} days present ({actual_days/expected_days*100:.1f}%)")
    
    if actual_days < expected_days * 0.8:
        print("⚠️  WARNING: Final month appears incomplete (partial period)")
    else:
        print("✓ Final month appears complete")